<a href="https://colab.research.google.com/github/chaithu-gif/AI_LAB/blob/main/week9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import re
from collections import deque
from dataclasses import dataclass
from typing import Dict, List, Optional, Set, Tuple, Union


# ==========================================
# 1. AST Nodes for Terms & FOL Formulas
# ==========================================

@dataclass(frozen=True)
class Var:
    name: str
    def __str__(self) -> str:
        return self.name


@dataclass(frozen=True)
class Const:
    name: str
    def __str__(self) -> str:
        return self.name


@dataclass(frozen=True)
class Func:
    name: str
    args: Tuple["Term", ...]
    def __str__(self) -> str:
        return f"{self.name}({', '.join(str(a) for a in self.args)})"


Term = Union[Var, Const, Func]


class Formula:
    pass


@dataclass(frozen=True)
class Pred(Formula):
    name: str
    args: Tuple[Term, ...]
    def __str__(self) -> str:
        if not self.args:
            return self.name
        return f"{self.name}({', '.join(str(a) for a in self.args)})"


@dataclass(frozen=True)
class Not(Formula):
    operand: Formula
    def __str__(self) -> str:
        if isinstance(self.operand, Pred):
            return f"~{self.operand}"
        return f"~({self.operand})"


@dataclass(frozen=True)
class And(Formula):
    left: Formula
    right: Formula
    def __str__(self) -> str:
        return f"({self.left} & {self.right})"


@dataclass(frozen=True)
class Or(Formula):
    left: Formula
    right: Formula
    def __str__(self) -> str:
        return f"({self.left} v {self.right})"


@dataclass(frozen=True)
class Implies(Formula):
    left: Formula
    right: Formula
    def __str__(self) -> str:
        return f"({self.left} -> {self.right})"


@dataclass(frozen=True)
class Iff(Formula):
    left: Formula
    right: Formula
    def __str__(self) -> str:
        return f"({self.left} <-> {self.right})"


@dataclass(frozen=True)
class ForAll(Formula):
    var: str
    body: Formula
    def __str__(self) -> str:
        return f"forall {self.var} ({self.body})"


@dataclass(frozen=True)
class Exists(Formula):
    var: str
    body: Formula
    def __str__(self) -> str:
        return f"exists {self.var} ({self.body})"


@dataclass(frozen=True)
class Literal:
    predicate: str
    args: Tuple[Term, ...]
    negated: bool = False

    def negate(self) -> "Literal":
        return Literal(self.predicate, self.args, not self.negated)

    def __str__(self) -> str:
        prefix = "~" if self.negated else ""
        if not self.args:
            return f"{prefix}{self.predicate}"
        return f"{prefix}{self.predicate}({', '.join(str(a) for a in self.args)})"


Clause = Tuple[Literal, ...]
Substitution = Dict[Var, Term]


def format_clause(clause: Clause) -> str:
    if not clause:
        return "[] (Empty Clause / Contradiction)"
    return " v ".join(str(lit) for lit in clause)


# ==========================================
# 2. English Sentence -> FOL Translator
# ==========================================

def english_to_fol(sentence: str) -> Optional[str]:
    """
    Converts controlled English sentences into formal FOL syntax.
    Returns None if the input is already written in FOL syntax.
    """
    s = sentence.strip().rstrip(".")
    if any(sym in s for sym in ["(", ")", "->", "<->", "~", "∀", "∃", "&", "|"]) or re.match(
        r"^(forall|exists)\b", s, re.IGNORECASE
    ):
        return None

    low = s.lower()

    # 1. "<Person> likes all kind(s) of <Category>"
    m = re.match(r"^([A-Za-z]+)\s+likes\s+all\s+kinds?\s+of\s+([A-Za-z]+)$", s, re.IGNORECASE)
    if m:
        person, cat = m.group(1).capitalize(), m.group(2).capitalize()
        return f"forall x ({cat}(x) -> Likes({person}, x))"

    # 2. "<A> and <B> are <Category>" (e.g., "Apple and vegetables are food")
    m = re.match(r"^([A-Za-z]+)\s+and\s+([A-Za-z]+)\s+are\s+([A-Za-z]+)$", s, re.IGNORECASE)
    if m:
        a, b, cat = m.group(1).capitalize(), m.group(2).capitalize(), m.group(3).capitalize()
        return f"{cat}({a}) & {cat}({b})"

    # 3. "Anything anyone <verb>s and not <adj> is <Category>"
    m = re.match(
        r"^anything\s+anyone\s+([a-z]+?)s?\s+and\s+(?:is\s+)?not\s+([a-z]+)\s+is\s+([a-z]+)$",
        low,
    )
    if m:
        verb = m.group(1).capitalize() + "s" if not m.group(1).endswith("s") else m.group(1).capitalize()
        adj = m.group(2).capitalize()
        cat = m.group(3).capitalize()
        return f"forall x forall y (({verb}(y, x) & ~{adj}(y)) -> {cat}(x))"

    # 4. "<Person> <verb>s <Object> and (is/still) <Adj>"
    m = re.match(
        r"^([A-Za-z]+)\s+([a-z]+?s)\s+([A-Za-z]+)\s+and\s+(?:is\s+|still\s+)?([a-z]+)$",
        s,
        re.IGNORECASE,
    )
    if m:
        person = m.group(1).capitalize()
        verb = m.group(2).capitalize()
        obj = m.group(3).capitalize()
        adj = m.group(4).capitalize()
        return f"{verb}({person}, {obj}) & {adj}({person})"

    # 5. "<Person1> <verb>s everything that <Person2> <verb>s"
    m = re.match(
        r"^([A-Za-z]+)\s+([a-z]+?s)\s+everything\s+(?:that\s+)?([A-Za-z]+)\s+([a-z]+?s)$",
        s,
        re.IGNORECASE,
    )
    if m:
        p1, v1 = m.group(1).capitalize(), m.group(2).capitalize()
        p2, v2 = m.group(3).capitalize(), m.group(4).capitalize()
        return f"forall x ({v2}({p2}, x) -> {v1}({p1}, x))"

    # 6. "Anyone who is [not] <A> implies [not] <B>"
    m = re.match(
        r"^anyone\s+who\s+is\s+(not\s+)?([a-z]+)\s+implies\s+(not\s+)?([a-z]+)$",
        low,
    )
    if m:
        neg1 = "~" if m.group(1) else ""
        pred1 = m.group(2).capitalize()
        neg2 = "~" if m.group(3) else ""
        pred2 = m.group(4).capitalize()
        return f"forall x ({neg1}{pred1}(x) -> {neg2}{pred2}(x))"

    # 7. "All/Every <A> is/are [not] <B>"
    m = re.match(r"^(?:all|every)\s+([a-z]+?)(?:s)?\s+(?:is|are)\s+(not\s+)?([a-z]+)$", low)
    if m:
        pred1 = m.group(1).capitalize()
        neg2 = "~" if m.group(2) else ""
        pred2 = m.group(3).capitalize()
        return f"forall x ({pred1}(x) -> {neg2}{pred2}(x))"

    # 8. "<Subj> is [not] [a/an] <Pred>"
    m = re.match(r"^([A-Za-z]+)\s+is\s+(not\s+)?(?:a\s+|an\s+)?([a-z]+)$", s, re.IGNORECASE)
    if m:
        subj = m.group(1).capitalize()
        neg = "~" if m.group(2) else ""
        pred = m.group(3).capitalize()
        return f"{neg}{pred}({subj})"

    # 9. "<Subj> [does not] <verb> <Obj>" (e.g., "John likes peanuts")
    m = re.match(r"^([A-Za-z]+)\s+(does\s+not\s+)?([a-z]+)\s+([A-Za-z]+)$", s, re.IGNORECASE)
    if m:
        subj = m.group(1).capitalize()
        neg = "~" if m.group(2) else ""
        raw_verb = m.group(3).capitalize()
        if neg and not raw_verb.endswith("s"):
            raw_verb += "s"
        obj = m.group(4).capitalize()
        return f"{neg}{raw_verb}({subj}, {obj})"

    raise ValueError(
        f"Unrecognized English pattern: '{sentence}'. "
        "Use a supported English pattern or enter formal FOL syntax (e.g., 'forall x (P(x) -> Q(x))')."
    )


# ==========================================
# 3. Recursive-Descent Parser for FOL Formulas
# ==========================================

TOKEN_SPEC = [
    ("IFF", r"<->|<=>|↔"),
    ("IMPLIES", r"->|=>|→"),
    ("AND", r"&|\^|∧|\bAND\b|\band\b"),
    ("OR", r"\||∨|\bOR\b|\bor\b|\bv\b"),
    ("NOT", r"~|!|¬|\bNOT\b|\bnot\b"),
    ("FORALL", r"∀|\bFORALL\b|\bforall\b"),
    ("EXISTS", r"∃|\bEXISTS\b|\bexists\b"),
    ("LPAREN", r"\("),
    ("RPAREN", r"\)"),
    ("COMMA", r","),
    ("IDENT", r"[A-Za-z_][A-Za-z0-9_]*"),
    ("SKIP", r"\s+"),
]
MASTER_REGEX = re.compile("|".join(f"(?P<{name}>{pat})" for name, pat in TOKEN_SPEC))


@dataclass
class Token:
    kind: str
    value: str


def tokenize(text: str) -> List[Token]:
    tokens: List[Token] = []
    pos = 0
    while pos < len(text):
        m = MASTER_REGEX.match(text, pos)
        if not m:
            raise ValueError(f"Unexpected character '{text[pos]}' at position {pos}")
        kind = m.lastgroup
        assert kind is not None
        if kind != "SKIP":
            tokens.append(Token(kind, m.group(kind)))
        pos = m.end()
    return tokens


class FOLParser:
    def __init__(self, tokens: List[Token]):
        self.tokens = tokens
        self.pos = 0

    def peek(self) -> Optional[Token]:
        return self.tokens[self.pos] if self.pos < len(self.tokens) else None

    def consume(self, expected_kind: Optional[str] = None) -> Token:
        tok = self.peek()
        if tok is None:
            raise ValueError("Unexpected end of formula.")
        if expected_kind and tok.kind != expected_kind:
            raise ValueError(f"Expected {expected_kind}, got '{tok.value}'")
        self.pos += 1
        return tok

    def parse(self) -> Formula:
        if not self.tokens:
            raise ValueError("Empty formula.")
        expr = self.parse_iff()
        if self.peek() is not None:
            raise ValueError(f"Unexpected trailing token: '{self.peek().value}'")
        return expr

    def parse_iff(self) -> Formula:
        left = self.parse_implies()
        while self.peek() and self.peek().kind == "IFF":
            self.consume("IFF")
            left = Iff(left, self.parse_implies())
        return left

    def parse_implies(self) -> Formula:
        left = self.parse_or()
        if self.peek() and self.peek().kind == "IMPLIES":
            self.consume("IMPLIES")
            return Implies(left, self.parse_implies())
        return left

    def parse_or(self) -> Formula:
        left = self.parse_and()
        while self.peek() and self.peek().kind == "OR":
            self.consume("OR")
            left = Or(left, self.parse_and())
        return left

    def parse_and(self) -> Formula:
        left = self.parse_unary()
        while self.peek() and self.peek().kind == "AND":
            self.consume("AND")
            left = And(left, self.parse_unary())
        return left

    def parse_unary(self) -> Formula:
        tok = self.peek()
        if tok is None:
            raise ValueError("Unexpected end of formula.")
        if tok.kind == "NOT":
            self.consume("NOT")
            return Not(self.parse_unary())
        if tok.kind in ("FORALL", "EXISTS"):
            q_kind = self.consume().kind
            var_tok = self.consume("IDENT")
            body = self.parse_unary()
            return ForAll(var_tok.value, body) if q_kind == "FORALL" else Exists(var_tok.value, body)
        if tok.kind == "LPAREN":
            self.consume("LPAREN")
            expr = self.parse_iff()
            self.consume("RPAREN")
            return expr
        if tok.kind == "IDENT":
            pred_name = self.consume("IDENT").value
            args: Tuple[Term, ...] = ()
            if self.peek() and self.peek().kind == "LPAREN":
                self.consume("LPAREN")
                arg_list: List[Term] = []
                if self.peek() and self.peek().kind != "RPAREN":
                    arg_list.append(self.parse_term())
                    while self.peek() and self.peek().kind == "COMMA":
                        self.consume("COMMA")
                        arg_list.append(self.parse_term())
                self.consume("RPAREN")
                args = tuple(arg_list)
            return Pred(pred_name, args)
        raise ValueError(f"Unexpected token '{tok.value}'")

    def parse_term(self) -> Term:
        ident = self.consume("IDENT").value
        if self.peek() and self.peek().kind == "LPAREN":
            self.consume("LPAREN")
            args: List[Term] = []
            if self.peek() and self.peek().kind != "RPAREN":
                args.append(self.parse_term())
                while self.peek() and self.peek().kind == "COMMA":
                    self.consume("COMMA")
                    args.append(self.parse_term())
            self.consume("RPAREN")
            return Func(ident, tuple(args))
        return Var(ident) if ident[0].islower() else Const(ident)


def parse_fol_formula(text: str) -> Formula:
    return FOLParser(tokenize(text)).parse()


# ==========================================
# 4. FOL Sentence -> CNF Conversion Steps
# ==========================================

def eliminate_implications(f: Formula) -> Formula:
    if isinstance(f, Pred):
        return f
    if isinstance(f, Not):
        return Not(eliminate_implications(f.operand))
    if isinstance(f, And):
        return And(eliminate_implications(f.left), eliminate_implications(f.right))
    if isinstance(f, Or):
        return Or(eliminate_implications(f.left), eliminate_implications(f.right))
    if isinstance(f, Implies):
        return Or(Not(eliminate_implications(f.left)), eliminate_implications(f.right))
    if isinstance(f, Iff):
        a, b = eliminate_implications(f.left), eliminate_implications(f.right)
        return And(Or(Not(a), b), Or(Not(b), a))
    if isinstance(f, ForAll):
        return ForAll(f.var, eliminate_implications(f.body))
    if isinstance(f, Exists):
        return Exists(f.var, eliminate_implications(f.body))
    return f


def move_not_inward(f: Formula) -> Formula:
    if isinstance(f, Pred):
        return f
    if isinstance(f, And):
        return And(move_not_inward(f.left), move_not_inward(f.right))
    if isinstance(f, Or):
        return Or(move_not_inward(f.left), move_not_inward(f.right))
    if isinstance(f, ForAll):
        return ForAll(f.var, move_not_inward(f.body))
    if isinstance(f, Exists):
        return Exists(f.var, move_not_inward(f.body))
    if isinstance(f, Not):
        inner = f.operand
        if isinstance(inner, Pred):
            return f
        if isinstance(inner, Not):
            return move_not_inward(inner.operand)
        if isinstance(inner, And):
            return Or(move_not_inward(Not(inner.left)), move_not_inward(Not(inner.right)))
        if isinstance(inner, Or):
            return And(move_not_inward(Not(inner.left)), move_not_inward(Not(inner.right)))
        if isinstance(inner, ForAll):
            return Exists(inner.var, move_not_inward(Not(inner.body)))
        if isinstance(inner, Exists):
            return ForAll(inner.var, move_not_inward(Not(inner.body)))
    return f


def subst_term(t: Term, mapping: Dict[str, Term]) -> Term:
    if isinstance(t, Var):
        return mapping.get(t.name, t)
    if isinstance(t, Func):
        return Func(t.name, tuple(subst_term(a, mapping) for a in t.args))
    return t


def subst_formula(f: Formula, mapping: Dict[str, Term]) -> Formula:
    if isinstance(f, Pred):
        return Pred(f.name, tuple(subst_term(a, mapping) for a in f.args))
    if isinstance(f, Not):
        return Not(subst_formula(f.operand, mapping))
    if isinstance(f, And):
        return And(subst_formula(f.left, mapping), subst_formula(f.right, mapping))
    if isinstance(f, Or):
        return Or(subst_formula(f.left, mapping), subst_formula(f.right, mapping))
    if isinstance(f, ForAll):
        return ForAll(f.var, subst_formula(f.body, {k: v for k, v in mapping.items() if k != f.var}))
    if isinstance(f, Exists):
        return Exists(f.var, subst_formula(f.body, {k: v for k, v in mapping.items() if k != f.var}))
    return f


_std_counter = 0


def standardize_variables(f: Formula, env: Optional[Dict[str, str]] = None) -> Formula:
    global _std_counter
    if env is None:
        env = {}
    if isinstance(f, Pred):
        return Pred(f.name, tuple(subst_term(a, {k: Var(v) for k, v in env.items()}) for a in f.args))
    if isinstance(f, Not):
        return Not(standardize_variables(f.operand, env))
    if isinstance(f, And):
        return And(standardize_variables(f.left, env), standardize_variables(f.right, env))
    if isinstance(f, Or):
        return Or(standardize_variables(f.left, env), standardize_variables(f.right, env))
    if isinstance(f, ForAll):
        _std_counter += 1
        new_var = f"{f.var}{_std_counter}"
        return ForAll(new_var, standardize_variables(f.body, {**env, f.var: new_var}))
    if isinstance(f, Exists):
        _std_counter += 1
        new_var = f"{f.var}{_std_counter}"
        return Exists(new_var, standardize_variables(f.body, {**env, f.var: new_var}))
    return f


_skolem_counter = 0


def skolemize_and_drop_quantifiers(f: Formula, univ_vars: Optional[List[str]] = None) -> Formula:
    global _skolem_counter
    if univ_vars is None:
        univ_vars = []
    if isinstance(f, (Pred, Not)):
        return f
    if isinstance(f, And):
        return And(
            skolemize_and_drop_quantifiers(f.left, univ_vars),
            skolemize_and_drop_quantifiers(f.right, univ_vars),
        )
    if isinstance(f, Or):
        return Or(
            skolemize_and_drop_quantifiers(f.left, univ_vars),
            skolemize_and_drop_quantifiers(f.right, univ_vars),
        )
    if isinstance(f, ForAll):
        return skolemize_and_drop_quantifiers(f.body, univ_vars + [f.var])
    if isinstance(f, Exists):
        _skolem_counter += 1
        sk_term: Term = (
            Func(f"sk{_skolem_counter}", tuple(Var(v) for v in univ_vars))
            if univ_vars
            else Const(f"SK{_skolem_counter}")
        )
        return skolemize_and_drop_quantifiers(subst_formula(f.body, {f.var: sk_term}), univ_vars)
    return f


def distribute_or_over_and(f: Formula) -> Formula:
    if isinstance(f, And):
        return And(distribute_or_over_and(f.left), distribute_or_over_and(f.right))
    if isinstance(f, Or):
        a = distribute_or_over_and(f.left)
        b = distribute_or_over_and(f.right)
        if isinstance(a, And):
            return And(distribute_or_over_and(Or(a.left, b)), distribute_or_over_and(Or(a.right, b)))
        if isinstance(b, And):
            return And(distribute_or_over_and(Or(a, b.left)), distribute_or_over_and(Or(a, b.right)))
        return Or(a, b)
    return f


def extract_clauses(f: Formula) -> List[Clause]:
    if isinstance(f, And):
        return extract_clauses(f.left) + extract_clauses(f.right)

    def collect_literals(node: Formula) -> List[Literal]:
        if isinstance(node, Or):
            return collect_literals(node.left) + collect_literals(node.right)
        if isinstance(node, Pred):
            return [Literal(node.name, node.args, negated=False)]
        if isinstance(node, Not) and isinstance(node.operand, Pred):
            return [Literal(node.operand.name, node.operand.args, negated=True)]
        raise ValueError(f"Unexpected node during CNF extraction: {node}")

    unique_lits: List[Literal] = []
    seen: Set[Literal] = set()
    for lit in collect_literals(f):
        if lit not in seen:
            seen.add(lit)
            unique_lits.append(lit)
    return [tuple(unique_lits)]


def sentence_to_cnf(raw_input: str, show_steps: bool = True) -> List[Clause]:
    fol_str = english_to_fol(raw_input)
    if fol_str is not None:
        if show_steps:
            print(f"  [English -> FOL]  : {fol_str}")
        formula = parse_fol_formula(fol_str)
    else:
        formula = parse_fol_formula(raw_input)
        if show_steps:
            print(f"  [Parsed FOL]      : {formula}")

    step1 = eliminate_implications(formula)
    step2 = move_not_inward(step1)
    step3 = standardize_variables(step2)
    step4 = skolemize_and_drop_quantifiers(step3)
    step5 = distribute_or_over_and(step4)
    clauses = extract_clauses(step5)

    if show_steps:
        print(f"  [Eliminate ->]    : {step1}")
        print(f"  [Move ~ inward]   : {step2}")
        print(f"  [Skolem & Drop ∀] : {step4}")
        print(f"  [CNF Clauses]     : {[format_clause(c) for c in clauses]}\n")
    return clauses


# ==========================================
# 5. Unification & Resolution Engine
# ==========================================

def apply_sub_term(term: Term, theta: Substitution) -> Term:
    if isinstance(term, Var) and term in theta:
        return apply_sub_term(theta[term], theta)
    if isinstance(term, Func):
        return Func(term.name, tuple(apply_sub_term(a, theta) for a in term.args))
    return term


def apply_sub_literal(lit: Literal, theta: Substitution) -> Literal:
    return Literal(lit.predicate, tuple(apply_sub_term(a, theta) for a in lit.args), lit.negated)


def apply_sub_clause(clause: Clause, theta: Substitution) -> Clause:
    seen: Set[Literal] = set()
    result: List[Literal] = []
    for lit in clause:
        sub_lit = apply_sub_literal(lit, theta)
        if sub_lit not in seen:
            seen.add(sub_lit)
            result.append(sub_lit)
    return tuple(result)


def occurs_check(var: Var, term: Term, theta: Substitution) -> bool:
    term = apply_sub_term(term, theta)
    if var == term:
        return True
    if isinstance(term, Func):
        return any(occurs_check(var, arg, theta) for arg in term.args)
    return False


def unify(
    x: Union[Term, List[Term]],
    y: Union[Term, List[Term]],
    theta: Optional[Substitution] = None,
) -> Optional[Substitution]:
    if theta is None:
        theta = {}
    if isinstance(x, list) and isinstance(y, list):
        if len(x) != len(y):
            return None
        for xi, yi in zip(x, y):
            theta = unify(xi, yi, theta)
            if theta is None:
                return None
        return theta

    assert isinstance(x, (Var, Const, Func)) and isinstance(y, (Var, Const, Func))
    x = apply_sub_term(x, theta)
    y = apply_sub_term(y, theta)

    if x == y:
        return theta
    if isinstance(x, Var):
        return unify_var(x, y, theta)
    if isinstance(y, Var):
        return unify_var(y, x, theta)
    if isinstance(x, Func) and isinstance(y, Func) and x.name == y.name and len(x.args) == len(y.args):
        return unify(list(x.args), list(y.args), theta)
    return None


def unify_var(var: Var, val: Term, theta: Substitution) -> Optional[Substitution]:
    if var in theta:
        return unify(theta[var], val, theta)
    if isinstance(val, Var) and val in theta:
        return unify(var, theta[val], theta)
    if occurs_check(var, val, theta):
        return None
    return {**theta, var: val}


_res_var_counter = 0


def standardize_apart_clause(clause: Clause) -> Clause:
    global _res_var_counter
    _res_var_counter += 1
    mapping: Dict[str, Term] = {}

    def rename_term(t: Term) -> Term:
        if isinstance(t, Var):
            if t.name not in mapping:
                base = t.name.split("_")[0]
                mapping[t.name] = Var(f"{base}_{_res_var_counter}")
            return mapping[t.name]
        if isinstance(t, Func):
            return Func(t.name, tuple(rename_term(a) for a in t.args))
        return t

    return tuple(
        Literal(lit.predicate, tuple(rename_term(a) for a in lit.args), lit.negated)
        for lit in clause
    )


def resolve(c1: Clause, c2: Clause) -> List[Tuple[Clause, Substitution]]:
    c1 = standardize_apart_clause(c1)
    c2 = standardize_apart_clause(c2)
    resolvents = []

    for i, lit1 in enumerate(c1):
        for j, lit2 in enumerate(c2):
            if lit1.predicate == lit2.predicate and lit1.negated != lit2.negated:
                theta = unify(list(lit1.args), list(lit2.args))
                if theta is not None:
                    rem1 = [l for idx, l in enumerate(c1) if idx != i]
                    rem2 = [l for idx, l in enumerate(c2) if idx != j]
                    resolvents.append((apply_sub_clause(tuple(rem1 + rem2), theta), theta))
    return resolvents


def fol_resolution_prove(kb_clauses: List[Clause], negated_goal_clauses: List[Clause], goal_text: str) -> bool:
    clauses: List[Clause] = list(kb_clauses) + list(negated_goal_clauses)
    goal_start_idx = len(kb_clauses)

    parents: Dict[int, Optional[Tuple[int, int, Substitution]]] = {idx: None for idx in range(len(clauses))}
    clause_to_id: Dict[Clause, int] = {c: idx for idx, c in enumerate(clauses)}

    print("=== Final CNF Knowledge Base & Negated Goal ===")
    for idx, c in enumerate(clauses):
        tag = " (Negated Goal)" if idx >= goal_start_idx else ""
        print(f"C{idx + 1}: {format_clause(c)}{tag}")
    print()

    queue = deque(range(goal_start_idx, len(clauses)))
    visited_pairs = set()

    while queue:
        curr_idx = queue.popleft()
        curr_clause = clauses[curr_idx]

        for other_idx in range(len(clauses)):
            if curr_idx == other_idx:
                continue
            pair = tuple(sorted((curr_idx, other_idx)))
            if pair in visited_pairs:
                continue
            visited_pairs.add(pair)

            for resolvent, theta in resolve(curr_clause, clauses[other_idx]):
                if resolvent not in clause_to_id:
                    new_id = len(clauses)
                    clauses.append(resolvent)
                    clause_to_id[resolvent] = new_id
                    parents[new_id] = (curr_idx, other_idx, theta)
                    queue.append(new_id)

                if len(resolvent) == 0:
                    empty_id = clause_to_id[resolvent]
                    print("=== Resolution Proof Steps ===")
                    print_proof_tree(empty_id, clauses, parents)
                    print(f"\nResult: Contradiction reached! Therefore, '{goal_text}' is PROVED.")
                    return True

    print(f"Result: Could not prove '{goal_text}' from the given Knowledge Base.")
    return False


def print_proof_tree(
    clause_id: int,
    clauses: List[Clause],
    parents: Dict[int, Optional[Tuple[int, int, Substitution]]],
    printed: Optional[Set[int]] = None,
) -> None:
    if printed is None:
        printed = set()
    if clause_id in printed:
        return
    printed.add(clause_id)

    parent_info = parents[clause_id]
    if parent_info is not None:
        p1, p2, theta = parent_info
        print_proof_tree(p1, clauses, parents, printed)
        print_proof_tree(p2, clauses, parents, printed)
        sub_str = "{" + ", ".join(f"{k} -> {v}" for k, v in theta.items()) + "}" if theta else "{}"
        print(
            f"Resolve C{p1 + 1} [{format_clause(clauses[p1])}] "
            f"and C{p2 + 1} [{format_clause(clauses[p2])}] "
            f"with {sub_str}\n"
            f"   => C{clause_id + 1}: {format_clause(clauses[clause_id])}"
        )


# ==========================================
# 6. Interactive Sentence Input Driver
# ==========================================

def main() -> None:
    print("=== First-Order Logic (FOL) Resolution Theorem Prover ===")
    print("Enter premises as English sentences OR FOL formulas:")
    print("  • English example : John likes all kind of food.")
    print("  • FOL example     : forall x (Food(x) -> Likes(John, x))")
    print("Press Enter on an empty line when finished.\n")

    kb_clauses: List[Clause] = []
    sentence_num = 1

    while True:
        try:
            line = input(f"Premise {sentence_num}: ").strip()
            if not line:
                if not kb_clauses:
                    print("Please enter at least one premise sentence.")
                    continue
                break
            clauses = sentence_to_cnf(line, show_steps=True)
            kb_clauses.extend(clauses)
            sentence_num += 1
        except ValueError as err:
            print(f"  [Error] {err}\n")

    while True:
        try:
            goal_str = input("Enter the conclusion sentence to prove: ").strip()
            if not goal_str:
                continue
            print("\nConverting Negated Goal to CNF:")
            goal_fol = english_to_fol(goal_str) or goal_str
            negated_goal_clauses = sentence_to_cnf(f"~({goal_fol})", show_steps=True)
            break
        except ValueError as err:
            print(f"  [Error] {err}\n")

    fol_resolution_prove(kb_clauses, negated_goal_clauses, goal_str)


if __name__ == "__main__":
    main()

=== First-Order Logic (FOL) Resolution Theorem Prover ===
Enter premises as English sentences OR FOL formulas:
  • English example : John likes all kind of food.
  • FOL example     : forall x (Food(x) -> Likes(John, x))
Press Enter on an empty line when finished.

Premise 1: john likes all kind of food
  [English -> FOL]  : forall x (Food(x) -> Likes(John, x))
  [Eliminate ->]    : forall x ((~Food(x) v Likes(John, x)))
  [Move ~ inward]   : forall x ((~Food(x) v Likes(John, x)))
  [Skolem & Drop ∀] : (~Food(x1) v Likes(John, x1))
  [CNF Clauses]     : ['~Food(x1) v Likes(John, x1)']

Premise 2: Apple and Vegetable are food
  [English -> FOL]  : Food(Apple) & Food(Vegetable)
  [Eliminate ->]    : (Food(Apple) & Food(Vegetable))
  [Move ~ inward]   : (Food(Apple) & Food(Vegetable))
  [Skolem & Drop ∀] : (Food(Apple) & Food(Vegetable))
  [CNF Clauses]     : ['Food(Apple)', 'Food(Vegetable)']

Premise 3: Anything anyone eats and not killed is food
  [English -> FOL]  : forall x forall y 